In [5]:
from shared.main import *

In [ ]:
# ============================================================
# NLP feature NPZ loader
# ============================================================

from pathlib import Path
import os

import numpy as np
import pandas as pd

CORE_REPS = [
    "choice",
    "choice_diff",
    "choice_diff_local",
    "choice_diff_prev1",
    "choice_diff_prev3",
    "choice_diff_history",
    "running_mean",
    "running_variance",
    "change",
]

EMBEDDING_MODEL_NAME = "openai"
SENTIMENT_MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"

def _subject_npz_candidates(out_dir, sub_id):
    """
    Candidate feature NPZ paths for a subject id with/without `sub-` prefix.
    """
    out_dir = Path(out_dir)
    sid = str(sub_id).strip()
    sid_no_prefix = sid[4:] if sid.startswith("sub-") else sid

    labels = []
    for label in [sid, sid_no_prefix, f"sub-{sid_no_prefix}"]:
        if label and label not in labels:
            labels.append(label)

    if sid_no_prefix.isdigit():
        sid_int = int(sid_no_prefix)
        if 1 <= sid_int <= 9:
            label = f"sub-{sid_int:02d}"
            if label not in labels:
                labels.append(label)

    return [out_dir / f"{label}_choice_features.npz" for label in labels]

def find_subject_choice_feature_npz(out_dir, sub_id):
    """
    Return the first existing choice-feature NPZ path for sub_id.
    """
    candidates = _subject_npz_candidates(out_dir, sub_id)
    for path in candidates:
        if path.exists():
            return path
    return None

def load_subject_choice_features(path):
    """
    Load one subject choice-feature NPZ.

    Returns
    -------
    embeddings : dict[str, np.ndarray]
        `rep_name -> trials x features` arrays.
    sentiment : pd.DataFrame or None
        Columns are positive, negative, neutral, compound when present.
    meta : dict
        Model names, rep names, path, and optional trial metadata.
    """
    path = Path(path)

    with np.load(path, allow_pickle=True) as z:
        files = set(z.files)
        reps = z["rep_names"].astype(str).tolist() if "rep_names" in files else []

        embeddings = {
            rep: np.asarray(z[f"embedding_{rep}"], dtype=np.float32)
            for rep in reps
            if f"embedding_{rep}" in files
        }

        sent_cols = ["positive", "negative", "neutral", "compound"]
        if any(f"sentiment_{col}" in files for col in sent_cols):
            sentiment = pd.DataFrame(
                {
                    col: np.asarray(z[f"sentiment_{col}"], dtype=np.float32)
                    for col in sent_cols
                    if f"sentiment_{col}" in files
                }
            )
        else:
            sentiment = None

        meta = {
            "path": str(path),
            "sub_id": str(z["sub_id"]) if "sub_id" in files else None,
            "embedding_model": str(z["embedding_model"]) if "embedding_model" in files else None,
            "sentiment_model": str(z["sentiment_model"]) if "sentiment_model" in files else None,
            "rep_names": reps,
        }

        for key in [
            "trial_order",
            "character_role_num",
            "chosen_text",
            "unchosen_text",
            "prevslide_text",
            "prior_same_character_choices",
        ]:
            if key in files:
                meta[key] = np.asarray(z[key])

    return embeddings, sentiment, meta

def attach_choice_features_to_subject_data(
    subject_data,
    feature_dir,
    *,
    required=True,
    validate_lengths=True,
):
    """
    Attach precomputed NLP feature NPZs to an existing subject_data dict.

    This does not compute embeddings or sentiment. It only reads the files made by
    `make_nlp_features.ipynb` and adds them under each subject entry.
    """
    feature_dir = Path(feature_dir)
    summary_rows = []

    for sub_key, sd in subject_data.items():
        if sd is None:
            continue

        sub_ids_to_try = [sd.get("sub_id", sub_key), sub_key]
        path = None

        for sid in sub_ids_to_try:
            path = find_subject_choice_feature_npz(feature_dir, sid)
            if path is not None:
                break

        if path is None:
            checked = []
            for sid in sub_ids_to_try:
                checked.extend(str(p) for p in _subject_npz_candidates(feature_dir, sid))

            msg = (
                f"No choice-feature NPZ found for subject {sub_key}. Checked:\n"
                + "\n".join(f"  - {p}" for p in dict.fromkeys(checked))
            )

            if required:
                raise FileNotFoundError(msg)

            sd["embeddings"] = {}
            sd["sentiment"] = None
            sd["choice_feature_path"] = None
            sd["choice_feature_meta"] = None
            sd["has_sem"] = False
            summary_rows.append(
                {
                    "sub_id": sub_key,
                    "status": "missing",
                    "n_trials": len(sd.get("behavior", [])),
                    "n_reps": 0,
                    "path": None,
                }
            )
            continue

        embeddings, sentiment, meta = load_subject_choice_features(path)
        n_beh = len(sd["behavior"])

        if validate_lengths:
            for rep, arr in embeddings.items():
                if arr.shape[0] != n_beh:
                    raise ValueError(
                        f"{sub_key} {rep}: feature rows={arr.shape[0]} "
                        f"but behavior rows={n_beh} ({path})"
                    )

            if sentiment is not None and len(sentiment) != n_beh:
                raise ValueError(
                    f"{sub_key} sentiment rows={len(sentiment)} "
                    f"but behavior rows={n_beh} ({path})"
                )

        sd["embeddings"] = embeddings
        sd["sentiment"] = sentiment
        sd["embedding_model"] = meta.get("embedding_model")
        sd["embedding_model_name"] = meta.get("embedding_model")
        sd["sentiment_model"] = meta.get("sentiment_model")
        sd["choice_feature_path"] = str(path)
        sd["choice_feature_meta"] = meta
        sd["has_sem"] = bool(embeddings)

        summary_rows.append(
            {
                "sub_id": sub_key,
                "status": "loaded",
                "n_trials": n_beh,
                "n_reps": len(embeddings),
                "rep_names": list(embeddings.keys()),
                "has_sentiment": sentiment is not None,
                "path": str(path),
            }
        )

    return pd.DataFrame(summary_rows)

# ============================================================
# Behavior loader
# ============================================================

def load_behavior(
    sub_id,
    neutrals=True,
    *,
    on_missing="none",
    data_root=None,
):
    """
    Load behavior dataframe for a given subject.
    """

    if on_missing not in {"none", "empty", "raise"}:
        raise ValueError("on_missing must be one of {'none', 'empty', 'raise'}")

    if data_root is not None:
        data_base = Path(data_root).expanduser().resolve()

    elif "data_dir" in globals():
        data_base = Path(globals()["data_dir"]).expanduser().resolve()

    elif "PROJECT_ROOT" in globals():
        data_base = Path(globals()["PROJECT_ROOT"]).expanduser().resolve() / "data"

    else:
        raise RuntimeError(
            "Could not resolve data directory. Define `data_dir`, define `PROJECT_ROOT`, "
            "or pass `data_root='/path/to/project/data'."
        )

    def _strip_prefix(x) -> str:
        s = str(x).strip()
        return s[4:] if s.startswith("sub-") else s

    def _parse_int(s: str):
        s = str(s).strip()
        return int(s) if s.isdigit() else None

    def _format_sid(x) -> str:
        raw = _strip_prefix(x)
        iv = _parse_int(raw)

        if iv is not None:
            core = f"{iv:02d}" if 1 <= iv <= 9 else str(iv)
        else:
            core = raw

        return f"sub-{core}"

    sid = _format_sid(sub_id)

    raw = _strip_prefix(sub_id)
    iv = _parse_int(raw)

    if iv is not None and 1 <= iv <= 100:
        beh_dir = data_base / "other-samples" / "tavares" / "preprocessed" / "behavior"

    elif iv is None:
        beh_dir = data_base / "other-samples" / "online" / "preprocessed" / "behavior"

    else:
        beh_dir = data_base / "preprocessed" / "behavior"

    fpath = beh_dir / f"{sid}.xlsx"

    if not fpath.exists():
        if on_missing == "raise":
            raise FileNotFoundError(f"Behavior file not found: {fpath}")
        if on_missing == "empty":
            return pd.DataFrame()
        return None

    df = pd.read_excel(fpath)

    if not neutrals:
        if "character_role_num" in df.columns:
            role_col = "character_role_num"
        elif "char_role_num" in df.columns:
            role_col = "char_role_num"
        else:
            if on_missing == "raise":
                raise ValueError(
                    f"Behavior file has no role column: {fpath}. "
                    "Expected `character_role_num` or `char_role_num`."
                )
            if on_missing == "empty":
                return pd.DataFrame()
            return None

        df = df[df[role_col] != 6].reset_index(drop=True)

    return df

# ============================================================
# Subject loader
# ============================================================

def load_subject_data(
    sub_id,
    *,
    atlas=None,
    glm_dir=None,
    load_fmri=True,
    neutrals=True,
    beh_on_missing="none",
    do_dots_mapping=True,
    verbose=True,
    sub_id_width=2,
):
    """
    Load one subject's behavior and optional atlas ROI fMRI betas.

    Semantic embeddings are intentionally NOT loaded here.
    They are added later by add_roberta_core_embeddings().
    """

    def log(msg):
        if verbose:
            print(f"[load_subject_data sub={sub_id}] {msg}")

    def normalize_sub_id(sub_id):
        sid_raw = str(sub_id).strip()
        if sid_raw.startswith("sub-"):
            sid_raw = sid_raw[4:]

        is_numeric = sid_raw.isdigit()
        sid_int = int(sid_raw) if is_numeric else None
        sid_str = sid_raw.zfill(sub_id_width) if is_numeric else sid_raw
        sid_with_prefix = f"sub-{sid_str}"
        is_main_sample = bool(is_numeric and sid_int >= 100)

        return sid_raw, sid_str, sid_with_prefix, is_numeric, is_main_sample

    def get_role_col(behav):
        if "character_role_num" in behav.columns:
            return "character_role_num"
        if "char_role_num" in behav.columns:
            return "char_role_num"
        return None

    def find_subject_row(df, sid_raw, sid_str, sid_with_prefix, is_numeric):
        sid_col = df["sub_id"].astype(str)

        row = df[sid_col == str(sid_str)]

        if row.empty and is_numeric:
            row = df[sid_col == str(sid_raw)]

        if row.empty:
            row = df[sid_col == sid_with_prefix]

        return row

    def parse_atlas(atlas):
        import nibabel as nib

        if isinstance(atlas, str):
            atlas = pd.read_pickle(atlas) if atlas.endswith(".pkl") else nib.load(atlas)

        if isinstance(atlas, dict):
            atlas_img = atlas["image"]
            rois = atlas["rois"]

            if isinstance(rois, dict):
                atlas_codes = [int(k) for k in rois.keys()]
                atlas_labels = [str(v) for v in rois.values()]
            else:
                atlas_labels = list(rois)
                atlas_codes = list(range(1, len(atlas_labels) + 1))

            return atlas_img, atlas_codes, atlas_labels

        if isinstance(atlas, nib.Nifti1Image):
            atlas_img = atlas
            labels = np.unique(atlas_img.get_fdata().astype(int))
            labels = labels[labels > 0]
            atlas_codes = [int(i) for i in labels]
            atlas_labels = [f"ROI-{i}" for i in labels]

            return atlas_img, atlas_codes, atlas_labels

        raise ValueError(f"atlas must be dict, str, or Nifti1Image; got {type(atlas)}")

    def find_beta_path(glm_dir, sid_with_prefix):
        beta_candidates = [
            os.path.join(glm_dir, f"{sid_with_prefix}_decision_trials_beta.nii.gz"),
            os.path.join(glm_dir, sid_with_prefix, "beta_decisions.nii.gz"),
            os.path.join(glm_dir, sid_with_prefix, "beta_decisions_resampled.nii.gz"),
        ]

        beta_path = next((p for p in beta_candidates if os.path.exists(p)), None)

        if beta_path is None:
            msg = "Beta image not found. Checked:\n" + "\n".join(
                f"  - {p}" for p in beta_candidates
            )
            raise FileNotFoundError(msg)

        return beta_path

    def load_roi_betas(atlas, glm_dir, sid_with_prefix, T_beh):
        import nibabel as nib
        from nilearn import image

        atlas_img, atlas_codes, atlas_labels = parse_atlas(atlas)
        beta_path = find_beta_path(glm_dir, sid_with_prefix)

        log(f"Loading beta image: {beta_path}")

        beta_img = nib.load(beta_path)
        beta = beta_img.get_fdata().astype(float)

        if beta.ndim != 4:
            raise ValueError(f"Expected 4D beta image, got shape {beta.shape}")

        T_fmri = beta.shape[3]

        if T_beh > 0 and T_fmri != T_beh:
            raise ValueError(
                f"fMRI/behavior length mismatch for {sid_str}: "
                f"fmri={T_fmri}, behavior={T_beh}"
            )

        beta_flat = beta.reshape(-1, T_fmri)

        atlas_resamp = image.resample_to_img(
            atlas_img,
            beta_img,
            interpolation="nearest",
        )

        atlas_flat = atlas_resamp.get_fdata().astype(int).ravel()

        if atlas_flat.shape[0] != beta_flat.shape[0]:
            raise ValueError(
                f"Atlas/beta voxel mismatch after resampling for {sid_str}: "
                f"atlas voxels={atlas_flat.shape[0]}, beta voxels={beta_flat.shape[0]}"
            )

        roi_betas = {
            name: beta_flat[atlas_flat == int(code)].T
            for code, name in zip(atlas_codes, atlas_labels)
        }

        return beta_path, roi_betas

    # -------------------------
    # Normalize subject id
    # -------------------------

    sid_raw, sid_str, sid_with_prefix, is_numeric, is_main_sample = normalize_sub_id(sub_id)

    out = {"sub_id": sid_str}
    log(f"sid_str={sid_str}, main_sample={is_main_sample}")

    # -------------------------
    # Behavior
    # -------------------------

    try:
        behav = load_behavior(
            sid_str,
            neutrals=neutrals,
            on_missing=beh_on_missing,
        )
    except Exception as e:
        raise RuntimeError(f"Behavior loading raised for sub {sid_str}") from e

    if behav is None:
        log("Behavior is None -> returning None")
        return None

    out["behavior"] = behav
    T_beh = len(behav)

    role_col = get_role_col(behav)
    if role_col is None and T_beh > 0:
        raise ValueError(
            f"Missing role column for sub {sid_str} "
            "(expected character_role_num or char_role_num)"
        )

    out["char_roles"] = behav[role_col].to_numpy(int) if role_col else np.array([], int)

    # -------------------------
    # Dots
    # -------------------------

    out.update(
        {
            "dots": None,
            "dots_roles": None,
            "dots_affine": None,
        }
    )

    dots_df = globals().get("data" if is_main_sample else "data_online", None)

    if isinstance(dots_df, pd.DataFrame) and "sub_id" in dots_df.columns:
        row = find_subject_row(dots_df, sid_raw, sid_str, sid_with_prefix, is_numeric)

        if not row.empty:
            try:
                dots_arr = get_coords(row.iloc[0], which="dots", include_neutral=True)
                dots_by_char = np.asarray(dots_arr)[0]

                roles = ["first", "second", "assistant", "powerful", "boss", "neutral"]

                try:
                    global_roles = list(CHARACTERS)
                    if set(global_roles) == set(roles) and global_roles != roles:
                        idx = [roles.index(r) for r in global_roles]
                        dots_by_char = dots_by_char[idx]
                        roles = global_roles
                except Exception:
                    pass

                out["dots"] = dots_by_char
                out["dots_roles"] = roles

                if (
                    do_dots_mapping
                    and T_beh > 0
                    and {"affil_coord", "power_coord"} <= set(behav.columns)
                ):
                    beh_xy = behav[["affil_coord", "power_coord"]].to_numpy(float)
                    beh_dots, affine = transform_beh_to_dots(
                        beh_xy,
                        dots_by_char,
                        anchor="end",
                    )
                    behav[["affil_coord_in_dots", "power_coord_in_dots"]] = beh_dots
                    out["dots_affine"] = affine

            except Exception as e:
                log(f"Dots attach/mapping skipped: {e}")

    else:
        log("No dots df found: missing `data`/`data_online` or not a DataFrame -> skipping dots.")

    # -------------------------
    # Embeddings placeholder
    # -------------------------

    out["embeddings"] = {}
    out["embedding_model"] = None
    out["embedding_model_name"] = None
    out["has_sem"] = False

    # -------------------------
    # fMRI
    # -------------------------

    load_fmri = bool(load_fmri and atlas is not None and glm_dir is not None)

    if load_fmri:
        beta_path, roi_betas = load_roi_betas(
            atlas,
            glm_dir,
            sid_with_prefix,
            T_beh,
        )

        out["beta_path"] = beta_path
        out["roi_betas"] = roi_betas
        out["has_fmri"] = True

    else:
        out["roi_betas"] = None
        out["has_fmri"] = False

    log("Success.")
    return out

# ============================================================
# Run
# ============================================================

atlas = 'Schaefer100'
if atlas == 'Tavares':
    atlas_pkl = PROJECT_ROOT / "masks" / "atlases" / "Tavares2015_spheres_atlas.pkl"
    out_pkl   = PROJECT_ROOT / "analyses" / "lss_decision" / "subject_data_Tavares.pkl"
elif atlas == 'Schaefer100':
    atlas_pkl = PROJECT_ROOT / "masks" / "atlases" / "Schaefer100_HO-subcort25_1mm.pkl"
    out_pkl   = PROJECT_ROOT / "analyses" / "lss_decision" / "subject_data_Schaefer100.pkl"

glm_dir     = PROJECT_ROOT / "analyses" / "lss_decision" / "glms"
feature_dir = PROJECT_ROOT / "data" / "narratives" / "choice_features"
all_incl_subs = incl_subs

subject_data = {}

# add in the behavioral and fMRI data
for sub_id in tqdm(all_incl_subs, desc="Loading subject data"):
    try:
        sd = load_subject_data(
            sub_id,
            atlas=str(atlas_pkl),
            glm_dir=str(glm_dir),
            load_fmri=True,
            verbose=True,
        )

        if sd is not None:
            subject_data[sub_id] = sd

    except FileNotFoundError as e:
        print(f"[Missing files] {sub_id}: {e}")

    except Exception as e:
        print(f"[Error] {sub_id}: {e}")

# add in the NLP features
feature_summary = attach_choice_features_to_subject_data(
    subject_data,
    feature_dir,
    required=True,
    validate_lengths=True,
)

pickle_file(subject_data, str(out_pkl))

print(f"\nLoaded NLP feature NPZs from:\n{feature_dir}")
print(f"\nSaved subject_data with behavior, fMRI, dots, and NLP features to:\n{out_pkl}")
print(feature_summary.head())


Loading subject data:   0%|          | 0/79 [00:00<?, ?it/s]

[load_subject_data sub=18001] sid_str=18001, main_sample=True
[load_subject_data sub=18001] Loading beta image: /Users/matty_gee/Desktop/Social/SocialCUD/analyses/lss_decision/glms/sub-18001_decision_trials_beta.nii.gz


Loading subject data:   1%|▏         | 1/79 [00:02<03:16,  2.52s/it]

[load_subject_data sub=18001] Success.
[load_subject_data sub=18002] sid_str=18002, main_sample=True
[load_subject_data sub=18002] Loading beta image: /Users/matty_gee/Desktop/Social/SocialCUD/analyses/lss_decision/glms/sub-18002_decision_trials_beta.nii.gz


Loading subject data:   3%|▎         | 2/79 [00:04<03:08,  2.44s/it]

...

Loading subject data:  97%|█████████▋| 77/79 [02:34<00:01,  1.14it/s]

[load_subject_data sub=23019] Success.
[load_subject_data sub=23020] sid_str=23020, main_sample=True
[Missing files] 23020: Beta image not found. Checked:
  - /Users/matty_gee/Desktop/Social/SocialCUD/analyses/lss_decision/glms/sub-23020_decision_trials_beta.nii.gz
  - /Users/matty_gee/Desktop/Social/SocialCUD/analyses/lss_decision/glms/sub-23020/beta_decisions.nii.gz
  - /Users/matty_gee/Desktop/Social/SocialCUD/analyses/lss_decision/glms/sub-23020/beta_decisions_resampled.nii.gz
[load_subject_data sub=23023] sid_str=23023, main_sample=True
[load_subject_data sub=23023] Loading beta image: /Users/matty_gee/Desktop/Social/SocialCUD/analyses/lss_decision/glms/sub-23023_decision_trials_beta.nii.gz


Loading subject data: 100%|██████████| 79/79 [02:37<00:00,  1.99s/it]

[load_subject_data sub=23023] Success.



Loaded NLP feature NPZs from:
/Users/matty_gee/Desktop/Social/SocialCUD/data/narratives/choice_features

Saved subject_data with behavior, fMRI, dots, and NLP features to:
/Users/matty_gee/Desktop/Social/SocialCUD/analyses/lss_decision/subject_data_Schaefer100.pkl
   sub_id  status  n_trials  n_reps  \
0   18001  loaded        63       9   
1   18002  loaded        63       9   
2   18003  loaded        63       9   
3   18004  loaded        63       9   
4   18005  loaded        63       9   

                                           rep_names  has_sentiment  \
0  [choice, choice_diff, choice_diff_local, choic...           True   
1  [choice, choice_diff, choice_diff_local, choic...           True   
2  [choice, choice_diff, choice_diff_local, choic...           True   
3  [choice, choice_diff, choice_diff_local, choic...           True   
4  [choice, choice_diff, choice_diff_local, choic...           True   

                                                path  
0  /Users/matty_g